# L06 -- Graph Neural Network

Companion notebook for L06. Implements a 2-layer GCN in **plain PyTorch** (no PyG dependency) and trains a graph-level binary classifier on synthetic feeder data.

**Prerequisite**: L05. Slides: `L06_gnn.pdf`.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0)
print(f"torch {torch.__version__}")

## 1. GCN layer in plain PyTorch

For a single graph with adjacency $A$, node feature matrix $H$, GCN does:
$$H^{(l+1)} = \sigma(\tilde D^{-1/2} \tilde A \tilde D^{-1/2} H^{(l)} W^{(l)})$$
where $\tilde A = A + I$ and $\tilde D$ is its degree matrix.

In [ ]:
def normalize_adj(A):
    A_tilde = A + torch.eye(A.size(0))
    d = A_tilde.sum(dim=1)
    D_inv_sqrt = torch.diag(d.pow(-0.5))
    return D_inv_sqrt @ A_tilde @ D_inv_sqrt   # \tilde D^{-1/2} \tilde A \tilde D^{-1/2}

class GCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin = nn.Linear(in_dim, out_dim, bias=False)
    def forward(self, A_hat, H):
        return self.lin(A_hat @ H)

class GridGCN(nn.Module):
    def __init__(self, in_dim=3, hidden=16):
        super().__init__()
        self.g1 = GCNLayer(in_dim, hidden)
        self.g2 = GCNLayer(hidden, hidden)
        self.head = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(),
                                  nn.Linear(hidden, 1), nn.Sigmoid())
    def forward(self, A_hat, H):
        h = torch.relu(self.g1(A_hat, H))
        h = torch.relu(self.g2(A_hat, h))
        z = h.max(dim=0).values             # max-pool across nodes
        return self.head(z)

## 2. Synthetic feeder dataset

Each graph is a 4-bus radial feeder. Node features are random (V, P, Q). Label = unsafe (1) if any V < 0.95 or any P > 0.30.

In [ ]:
ADJ_4BUS = torch.tensor([[0,1,0,0],
                         [1,0,1,0],
                         [0,1,0,1],
                         [0,0,1,0]], dtype=torch.float32)
A_HAT = normalize_adj(ADJ_4BUS)

def sample_graph():
    V = 0.92 + 0.13 * torch.rand(4)
    P = 0.50 * torch.rand(4)
    Q = 0.25 * torch.rand(4)
    H = torch.stack([V, P, Q], dim=1)
    y = float((V.min() < 0.95) or (P.max() > 0.30))
    return H, torch.tensor([y])

torch.manual_seed(0)
train_graphs = [sample_graph() for _ in range(400)]
val_graphs   = [sample_graph() for _ in range(100)]
pos = sum(y.item() for _, y in train_graphs)
print(f"Training graphs: {len(train_graphs)} ({pos:.0f} positive, {len(train_graphs)-pos:.0f} negative)")

## 3. Train

In [ ]:
model = GridGCN()
opt = torch.optim.Adam(model.parameters(), lr=5e-3)
bce = nn.BCELoss()

def epoch_avg_loss(graphs, train=True):
    total = 0.0
    for H, y in graphs:
        y_hat = model(A_HAT, H).view(1)
        loss = bce(y_hat, y)
        if train:
            opt.zero_grad(); loss.backward(); opt.step()
        total += loss.item()
    return total / len(graphs)

tr_hist, va_hist = [], []
for epoch in range(40):
    model.train();  tr_hist.append(epoch_avg_loss(train_graphs, True))
    model.eval()
    with torch.no_grad(): va_hist.append(epoch_avg_loss(val_graphs, False))

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(tr_hist, label="train"); ax.plot(va_hist, label="val")
ax.set_xlabel("epoch"); ax.set_ylabel("BCE"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Validation recall at threshold 0.5
model.eval(); tp=fn=fp=tn=0
with torch.no_grad():
    for H, y in val_graphs:
        pred = float(model(A_HAT, H).item() >= 0.5)
        tp += (pred == 1 and y.item() == 1)
        fn += (pred == 0 and y.item() == 1)
        fp += (pred == 1 and y.item() == 0)
        tn += (pred == 0 and y.item() == 0)
print(f"TP={tp}, FN={fn}, FP={fp}, TN={tn}")
print(f"recall={tp/(tp+fn+1e-9):.3f}, precision={tp/(tp+fp+1e-9):.3f}")

## 4. Watch oversmoothing as depth increases

In [ ]:
class DeepGCN(nn.Module):
    def __init__(self, in_dim=3, hidden=16, L=8):
        super().__init__()
        layers = [GCNLayer(in_dim, hidden)] + [GCNLayer(hidden, hidden) for _ in range(L-1)]
        self.layers = nn.ModuleList(layers)
        self.head = nn.Linear(hidden, 1)
    def forward(self, A_hat, H, return_all=False):
        outs = []
        h = H
        for layer in self.layers:
            h = torch.relu(layer(A_hat, h))
            outs.append(h.clone())
        if return_all: return outs
        return torch.sigmoid(self.head(h.max(dim=0).values))

torch.manual_seed(0)
m_deep = DeepGCN(L=8)
H_demo, _ = sample_graph()
outs = m_deep(A_HAT, H_demo, return_all=True)
sims = [torch.nn.functional.cosine_similarity(h, h.mean(0, keepdim=True).expand_as(h)).mean().item()
        for h in outs]

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(range(1, len(sims)+1), sims, "o-")
ax.set_xlabel("layer"); ax.set_ylabel("mean cosine(h_v, mean(h))")
ax.set_title("Node embeddings collapse toward the mean (oversmoothing)")
ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

## Homework

### Required
1. Increase the synthetic graph size from 4 to 8 buses (keep it radial). Re-train. Does the GCN still learn the rule?
2. Replace `max`-pool with `mean`-pool. Does it help or hurt recall?
3. Compare to a flat MLP on the same task (concatenate the 4 nodes' features into a 12-vector). Which wins?

### Optional
- Add edge features (R, X) by replacing $A$ with a weighted adjacency.
- Reimplement using PyTorch Geometric's `GCNConv` and check you get the same accuracy.

In [ ]:
# TODO Q1: 8-bus radial
# ...

# TODO Q2: mean-pool replacement
# ...

# TODO Q3: flat MLP comparison
# ...